# 09_docstring

对应 `index.md` 第 9 阶段：docstring 怎么变成工具说明，以及模型到底拿到哪一份文字。

笔记本在 `codes/langchain/09_docstring/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langchain/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

模型：`init_chat_model("deepseek:...")`，密钥 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_MODEL`）。本阶段只围绕说明文字，不写工具的执行闭环，那部分见 `04_tools`。


In [3]:
import os
from pathlib import Path


def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "09_docstring" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "09_docstring"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here


def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value


ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))


(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/09_docstring'),
 True)

## 1. docstring 变成 description

用 `@tool` 定义函数 `celsius(f: float) -> float`（摄氏转华氏），docstring 只写一行：`摄氏转华氏。`。函数名和参数名就照题面写，不要另起。

依次打印 `celsius.name`、`celsius.description`，再打印 `celsius.description == celsius.func.__doc__.strip()`。


In [29]:
from langchain_core.tools import tool

# 作答

@tool
def celsius(f: float) -> float:
    """摄氏度转华氏度"""
    return f * 9 / 5 + 32

print(celsius.name)
print(celsius.description)
print(celsius.description == celsius.func.__doc__.strip())
#评阅
# 逻辑对：name 取函数名，description 取 docstring，比较返回 True。
# 但 docstring 写成了「摄氏度转华氏度」，题目指定的是「摄氏转华氏。」；这类题照题面抄。

#参考答案
# @tool
# def celsius(f: float) -> float:
#     """摄氏转华氏。"""
#     return f * 9 / 5 + 32
# print(celsius.name)
# print(celsius.description)
# print(celsius.description == celsius.func.__doc__.strip())


celsius
摄氏度转华氏度
True


## 2. 名字和说明可以被参数覆盖

已有普通函数 `mul`（docstring 是 `两数相乘。`）。

不要改这个函数，拿它做出一个工具：名字 `multiply_numbers`，说明 `把两个整数相乘。`。提示：`@tool(...)` 装饰器做的事就是 `tool(...)(函数)`。

打印工具的 `name`、`description`，再打印 `mul.__doc__`，说明这两处文字各自来自哪里。


In [31]:
from langchain_core.tools import tool


def mul(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b


# 作答

multiply_numbers = tool("multiply_numbers", description="把两个整数相乘。")(mul)

print(multiply_numbers.name)
print(multiply_numbers.description)
print(mul.__doc__)
#评阅
# 对。tool(...)(mul) 拿原函数建工具，没有另写一个函数把 mul 包起来。
# 三行打印正好把来源分开：name 来自 tool() 第一个参数，description 来自 description=，
# 而 mul.__doc__ 仍是「两数相乘。」，说明原函数没被改写。

#参考答案
# multiply_numbers = tool("multiply_numbers", description="把两个整数相乘。")(mul)
# print(multiply_numbers.name)
# print(multiply_numbers.description)
# print(mul.__doc__)


multiply_numbers
把两个整数相乘。
两数相乘。


## 3. parse_docstring 把 Args 段拆进 schema

已有普通函数 `unit_price`，它的 docstring 是 Google 风格：摘要是 `算单价。`，`Args:` 段里写了 `total: 总价`、`count: 件数`。

把它做成工具，要求 `Args:` 段不再留在工具的说明里，而是变成两个参数各自的 description。

打印工具的 `description` 和 `args`，回答：原来 `Args:` 段里那两句话跑到哪里去了？


In [33]:
from langchain_core.tools import tool


def unit_price(total: float, count: int) -> float:
    """算单价。

    Args:
        total: 总价
        count: 件数
    """
    return total / count

# 作答

unit_price_tool = tool("unit_price", description="算单价。", parse_docstring=True)(unit_price)

print(unit_price_tool.description)
print(unit_price_tool.args)
#评阅
# 对。Args 段的两句话落进了 args 里各自的 description，工具说明只剩摘要。
# name / description 两个参数在这里是多余的：函数本来就叫 unit_price，摘要也会自动取第一段。
# 回答：那两句话跑到 args_schema 的参数字段说明里了。

#参考答案
# unit_price_tool = tool(parse_docstring=True)(unit_price)
# print(unit_price_tool.description)
# print(unit_price_tool.args)


算单价。
{'total': {'description': '总价', 'title': 'Total', 'type': 'number'}, 'count': {'description': '件数', 'title': 'Count', 'type': 'integer'}}


## 4. 不是 Google 风格的 docstring

已有普通函数 `weighted`，docstring 用的是 NumPy 风格的 `Parameters` 段，不是 `Args:` 段。

1. 把它做成 `parse_docstring=True` 的工具，用 `try/except` 接住异常，打印异常类型和 `str(异常)`。
2. 再加 `error_on_invalid_docstring=False` 做一次，打印这次的 `description` 和 `args`。

回答：开关关掉以后，两个参数还有自己的说明吗？


In [37]:
from langchain_core.tools import tool


def weighted(score: float, weight: float) -> float:
    """算加权分。

    Parameters
    ----------
    score : float
        得分
    weight : float
        权重
    """
    return score * weight


# 作答

try:
    t = tool("weighted", description="算加权分。", parse_docstring=True)(weighted)
except Exception as e:
    print(type(e), str(e))

try:
    t = tool("weighted", description="算加权分。", parse_docstring=True, error_on_invalid_docstring=False)(weighted)
    
    print(t.description)
    print(t.args)
except Exception as e:
    print(type(e), str(e))
#评阅
# 两次都跑对了。第一次 ValueError: Found invalid Google-Style docstring.；
# 第二次不报错，args 只剩 title / type，没有 description。
# 回答：没有说明。这个开关只做「别抛异常」，解析器仍然只认 Args: 段，NumPy 的
# Parameters 段识别不了；而且那段原文会整段混进 description——你这里是因为显式传了
# description="算加权分。" 才挡住，去掉它，说明会变成整段 NumPy 文本。

#参考答案
# try:
#     tool("weighted", description="算加权分。", parse_docstring=True)(weighted)
# except ValueError as e:
#     print(type(e).__name__, e)
# t = tool("weighted", description="算加权分。", parse_docstring=True, error_on_invalid_docstring=False)(weighted)
# print(t.description)
# print(t.args)


<class 'ValueError'> Found invalid Google-Style docstring.
算加权分。
{'score': {'title': 'Score', 'type': 'number'}, 'weight': {'title': 'Weight', 'type': 'number'}}


## 5. Args 里的参数名对不上

已有普通函数 `ratio`，它的 `Args:` 段里写了一个签名里并不存在的参数 `scale`。

同样用 `try/except` 试两次：一次只给 `parse_docstring=True`，一次再加 `error_on_invalid_docstring=False`。每次都打印异常类型和 `str(异常)`。

回答：`error_on_invalid_docstring=False` 对这种错有用吗？


In [38]:
from langchain_core.tools import tool


def ratio(part: int, whole: int) -> float:
    """算占比。

    Args:
        part: 部分
        scale: 多写的，签名里没有这个参数
    """
    return part / whole


# 作答

try:
    t = tool("ratio", description="算占比。", parse_docstring=True)(ratio)
except Exception as e:
    print(type(e), str(e))

try:
    t = tool("ratio", description="算占比。", parse_docstring=True, error_on_invalid_docstring=False)(ratio)
    print(t.description)
    print(t.args)
except Exception as e:
    print(type(e), str(e))
#评阅
# 对。两次都是同一个错：Arg scale in docstring not found in function signature.
# 回答：没用。这个开关只管「格式是不是 Google 风格」那一条校验；
# 「Args 里的名字对不上签名」是另一条无条件跑的校验，关开关也拦不住。

#参考答案
# for flag in (True, False):
#     try:
#         tool("ratio", description="算占比。", parse_docstring=True, error_on_invalid_docstring=flag)(ratio)
#     except ValueError as e:
#         print(type(e).__name__, e)


<class 'ValueError'> Arg scale in docstring not found in function signature.
<class 'ValueError'> Arg scale in docstring not found in function signature.


## 6. 说明的两个去处

已有工具 `discount`（`parse_docstring=True`，docstring 有摘要和 `Args:` 段）和工具列表 `tools = [discount]`。

1. 用 `convert_to_openai_tool(discount)` 导出函数调用协议里的样子，打印 `function.description`，以及 `parameters["properties"]["rate"]["description"]`。
2. 用 `render_text_description(tools)` 渲染成一段文本，打印出来。

回答：`render_text_description` 渲染出的那行里还有 `Args:` 段的参数说明吗？模型真正收到的那份 schema 是哪一种？


In [44]:
from langchain_core.tools import render_text_description, tool
from langchain_core.utils.function_calling import convert_to_openai_tool


@tool(parse_docstring=True)
def discount(price: float, rate: float) -> float:
    """算折后价。

    Args:
        price: 原价
        rate: 折扣率，0 到 1 之间
    """
    return price * rate


tools = [discount]

# 作答

t = convert_to_openai_tool(discount)
print(t)
print(t["function"]["description"])
print(t["function"]["parameters"]["properties"]["rate"]["description"])

text = render_text_description(tools)
print(text)

#评阅
# 三处都取对了（顺带打印整个 dict 无妨）。
# 回答：render_text_description 渲染出的那行只有「签名 - 摘要」，没有 Args 段的参数说明；
# 模型真正收到的是 convert_to_openai_tool 导出的那份 function schema，
# 参数说明在 parameters.properties.<参数>.description 里。

#参考答案
# spec = convert_to_openai_tool(discount)
# print(spec["function"]["description"])
# print(spec["function"]["parameters"]["properties"]["rate"]["description"])
# print(render_text_description(tools))


{'type': 'function', 'function': {'name': 'discount', 'description': '算折后价。', 'parameters': {'properties': {'price': {'description': '原价', 'type': 'number'}, 'rate': {'description': '折扣率，0 到 1 之间', 'type': 'number'}}, 'required': ['price', 'rate'], 'type': 'object'}}}
算折后价。
折扣率，0 到 1 之间
discount(price: float, rate: float) -> float - 算折后价。


## 7. 真模型按说明选工具

已有两个工具：`add_numbers`（说明是两数相加）和 `product_numbers`（说明是两数相乘）。

用 `init_chat_model`（默认 `deepseek:deepseek-v4-flash`，或读 `DEEPSEEK_MODEL` 拼成 `deepseek:{name}`）建模型，对 `tools` 做 `bind_tools`，再 `invoke` 一条用户消息：`6 乘 7 等于多少？`。

打印返回消息的 `tool_calls`：应该有一条，`name` 是 `product_numbers`，`args` 里两个数是 6 和 7。


In [ ]:
from langchain_core.tools import tool


@tool
def add_numbers(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


@tool
def product_numbers(a: int, b: int) -> int:
    """两数相乘。"""
    return a * b


tools = [add_numbers, product_numbers]

# 作答

from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-v4-flash")

bound = model.bind_tools(tools)

response = bound.invoke("6 乘 7 等于多少？")

from rich import print as rprint
rprint(response.tool_calls)

#评阅
# 对。模型按 description 选出了 product_numbers(6, 7)。
# 两点：题面说模型名可从 DEEPSEEK_MODEL 读，你写死了 v4-flash；
# 这里 bind_tools 能过是因为没强制 tool_choice——换成 with_structured_output 的默认
# function_calling，v4-flash 会报 Thinking mode does not support this tool_choice。

#参考答案
# from langchain.chat_models import init_chat_model
# name = os.environ.get("DEEPSEEK_MODEL", "deepseek-v4-flash")
# ai = init_chat_model(f"deepseek:{name}").bind_tools(tools).invoke("6 乘 7 等于多少？")
# print(ai.tool_calls)


[{'name': 'product_numbers', 'args': {'a': 6, 'b': 7}, 'id': 'call_00_bPcQCU14MgxjSfUUhVLz2786', 'type': 'tool_call'}]
